# Neo4j Import

### Estimated time: 5-10 minutes

This notebook loads all workshop data into Neo4j in a single step:

1. **Structured data** (7 CSV files) loaded via the Neo4j Spark Connector as nodes and relationships
2. **Document graph** (14 HTML documents) with pre-computed vector embeddings loaded via the Neo4j Python driver

After this notebook runs, Neo4j contains the complete graph:
- **764 nodes** across 7 types (Customer, Account, Bank, Transaction, Position, Stock, Company)
- **814 relationships** across 7 types (HAS_ACCOUNT, AT_BANK, PERFORMS, BENEFITS_TO, HAS_POSITION, OF_SECURITY, OF_COMPANY)
- **14 Document nodes** and approximately 50-100 Chunk nodes with 1024-dimensional embedding vectors
- **Vector and full-text indexes** on Chunk nodes for hybrid search

### Prerequisites

- Run **0 - Required Setup** first
- Cluster must be **Dedicated** mode with the Neo4j Spark Connector Maven library installed

## Data Model

The structured graph models a retail investment domain:

```
Customer ──HAS_ACCOUNT──> Account ──AT_BANK──> Bank
                             │
                             ├──PERFORMS──> Transaction ──BENEFITS_TO──> Account
                             │
                             └──HAS_POSITION──> Position ──OF_SECURITY──> Stock ──OF_COMPANY──> Company
```

The document graph adds a layer of unstructured content:

```
Document ──DESCRIBES──> Customer
   ▲
   │
Chunk ──FROM_DOCUMENT──> Document
   │
Chunk ──NEXT_CHUNK──> Chunk
```

Each Chunk node stores a text segment and its vector embedding, enabling semantic search
across customer profiles, company analyses, investment guides, and regulatory documents.

## About the Pre-computed Embeddings

The document embeddings were generated ahead of time using the **databricks-gte-large-en**
foundation model (1024 dimensions). The process was:

1. Parse 14 HTML documents with BeautifulSoup to extract clean text
2. Split each document into chunks (4000 characters, 200 character overlap)
3. Generate a vector embedding for each chunk via the Databricks embedding endpoint
4. Save everything to a JSON file that ships with the workshop

The generation script lives in `full_demo/agent_modules/generate_embeddings.py` if you want to see
exactly how it works or regenerate the embeddings with a different model.

## Run the Import

In [ ]:
%run ./Includes/config

In [ ]:
%run ./Includes/_lib/neo4j_import

In [ ]:
run_full_import()

## Explore the Graph

Open Neo4j Browser and try these queries:

**See the full schema:**
```cypher
CALL db.schema.visualization()
```

**Count nodes by label:**
```cypher
MATCH (n)
RETURN labels(n)[0] AS label, count(n) AS count
ORDER BY count DESC
```

**Find a customer's portfolio:**
```cypher
MATCH (c:Customer {first_name: 'James', last_name: 'Anderson'})
      -[:HAS_ACCOUNT]->(a:Account)
      -[:HAS_POSITION]->(p:Position)
      -[:OF_SECURITY]->(s:Stock)
      -[:OF_COMPANY]->(co:Company)
RETURN c.first_name + ' ' + c.last_name AS customer,
       s.ticker, co.name, p.shares, p.current_value
```

**Search documents by vector similarity:**
```cypher
// Requires calling the embedding endpoint first, but you can browse chunks:
MATCH (c:Chunk)-[:FROM_DOCUMENT]->(d:Document)
WHERE d.document_type = 'customer_profile'
RETURN d.title, c.index, left(c.text, 200) AS preview
ORDER BY d.title, c.index
LIMIT 10
```

## Next Steps

The graph is fully loaded. Continue to:

- **4 - Neo4j to Lakehouse**: Export Neo4j data back to Delta Lake tables for use with Databricks AI agents
- **5 - AI Agents**: Create Genie and Knowledge Assistant agents
- **6 - Supervisor Agent**: Combine both agents into a unified multi-agent system
- **7 - Augmentation Agent**: Analyze gaps between structured data and documents with DSPy
- **8 - Graph Enrichment**: Write enrichment proposals back to Neo4j, closing the loop